# Dynamic Harmonic Regression

**Dynamic harmonic regression** uses Fourier terms as exogenous variables
in a SARIMAX model to capture complex or long-period seasonality.

The idea: instead of using seasonal ARIMA (which needs $m$ seasonal lags),
we represent the seasonal pattern with $K$ pairs of sine and cosine functions:

$$
s_t = \sum_{k=1}^{K} \left[ \alpha_k \sin\!\left(\frac{2\pi k t}{m}\right) + \beta_k \cos\!\left(\frac{2\pi k t}{m}\right) \right]
$$

These Fourier terms are passed as **exogenous variables** to SARIMAX, which
also models the remaining autocorrelation with an ARIMA error structure.

**Why this matters:**
- Seasonal ARIMA with $m=365$ (daily data, annual cycle) requires estimating
  hundreds of parameters — impractical
- Fourier terms can approximate any seasonal shape with just $2K$ parameters
- $K$ controls smoothness: small $K$ = smooth pattern, large $K$ = flexible

This notebook covers:
1. Fourier terms explained visually
2. Implementation of Fourier feature creation
3. Choosing $K$ — smoothness vs flexibility
4. SARIMAX with Fourier features
5. Comparison across different $K$ values

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox
from sklearn.metrics import mean_absolute_error, mean_squared_error

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["lines.linewidth"] = 1.5

DATA_DIR = Path("../data")

---
## 1. What Are Fourier Terms?

A **Fourier pair** at harmonic $k$ with period $m$ consists of:

$$
\sin\!\left(\frac{2\pi k t}{m}\right) \quad \text{and} \quad \cos\!\left(\frac{2\pi k t}{m}\right)
$$

- $k = 1$: one complete cycle per season (the fundamental frequency)
- $k = 2$: two cycles per season (first harmonic)
- $k = K$: $K$ cycles per season

The maximum useful $K$ is $m/2$ (Nyquist frequency).  For monthly data
with $m = 12$, the maximum is $K = 6$.

Larger $K$ gives more flexibility but risks overfitting.  Smaller $K$
gives a smoother seasonal pattern.

In [ ]:
# Visualise Fourier terms for m=12 (monthly seasonality)
t = np.arange(36)  # 3 years of monthly data
m = 12

fig, axes = plt.subplots(3, 2, figsize=(14, 10))
for k in range(1, 4):
    sin_vals = np.sin(2 * np.pi * k * t / m)
    cos_vals = np.cos(2 * np.pi * k * t / m)

    axes[k - 1, 0].plot(t, sin_vals, color="tab:blue")
    axes[k - 1, 0].set_title(f"$\\sin(2\\pi \\cdot {k} \\cdot t / {m})$  — k={k}")
    axes[k - 1, 0].axhline(0, color="grey", linestyle="--", alpha=0.5)
    axes[k - 1, 0].set_ylabel("Value")

    axes[k - 1, 1].plot(t, cos_vals, color="tab:orange")
    axes[k - 1, 1].set_title(f"$\\cos(2\\pi \\cdot {k} \\cdot t / {m})$  — k={k}")
    axes[k - 1, 1].axhline(0, color="grey", linestyle="--", alpha=0.5)
    axes[k - 1, 1].set_ylabel("Value")

for ax in axes[-1]:
    ax.set_xlabel("Time index t")

plt.suptitle("Fourier Terms for m=12", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("k=1: 1 cycle per 12 months — captures the broad annual shape")
print("k=2: 2 cycles per 12 months — captures 6-month sub-patterns")
print("k=3: 3 cycles per 12 months — captures finer quarterly detail")

---
## 2. Creating Fourier Features

In [ ]:
def create_fourier_features(index, period, K):
    """Create K Fourier sine/cosine pairs for a given period.

    Parameters
    ----------
    index : pd.DatetimeIndex
        The time index of the series.
    period : int
        The seasonal period (e.g., 12 for monthly, 7 for daily weekly).
    K : int
        Number of Fourier pairs.  Must be <= period // 2.

    Returns
    -------
    pd.DataFrame
        DataFrame with 2*K columns: sin_1, cos_1, sin_2, cos_2, ...
    """
    if K > period // 2:
        raise ValueError(f"K={K} exceeds maximum {period // 2} for period={period}")

    t = np.arange(len(index))
    features = {}
    for k in range(1, K + 1):
        features[f"sin_{k}"] = np.sin(2 * np.pi * k * t / period)
        features[f"cos_{k}"] = np.cos(2 * np.pi * k * t / period)
    return pd.DataFrame(features, index=index)


# Demo: create Fourier features for 36 months with K=2
demo_index = pd.date_range("2020-01-01", periods=36, freq="MS")
demo_fourier = create_fourier_features(demo_index, period=12, K=2)
print(f"Shape: {demo_fourier.shape}")
print(f"Columns: {list(demo_fourier.columns)}")
demo_fourier.head(13)

---
## 3. Visualising What Different K Values Capture

Let us see how a linear combination of Fourier terms with different $K$
can approximate various seasonal shapes.

In [ ]:
# Show how increasing K produces more flexible seasonal shapes
# Using a synthetic example: sharp peak in December
t = np.arange(24)  # 2 years
m = 12

# Synthetic "true" seasonal pattern (sharp December peak)
true_season = np.where(t % 12 == 11, 5.0, 0.0)  # spike at month 12
true_season += np.sin(2 * np.pi * t / 12) * 0.5  # plus gentle wave

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
K_values = [1, 2, 3, 4, 5, 6]

for ax, K in zip(axes.flat, K_values):
    fourier = create_fourier_features(pd.RangeIndex(24), period=12, K=K)
    # Fit a simple OLS to find the best Fourier approximation
    from numpy.linalg import lstsq
    coeffs, _, _, _ = lstsq(fourier.values, true_season, rcond=None)
    approx = fourier.values @ coeffs

    ax.bar(t, true_season, alpha=0.3, color="tab:blue", label="True pattern")
    ax.plot(t, approx, color="tab:red", linewidth=2, label=f"K={K} Fourier")
    ax.set_title(f"K = {K}  ({2*K} terms)")
    ax.legend(fontsize=8)
    ax.set_ylim(-2, 7)

plt.suptitle("Fourier Approximation of a Sharp Seasonal Peak", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("K=1: too smooth — cannot capture the December spike.")
print("K=6: fully flexible — reproduces any 12-period pattern exactly.")
print("K=3-4: a reasonable middle ground for most real seasonal data.")

---
## 4. Load Energy Production Data

We will use the **Energy Production Index** — a monthly series with a clear
annual seasonal pattern, ideal for demonstrating dynamic harmonic regression.

In [ ]:
energy = pd.read_csv(
    DATA_DIR / "EnergyProduction.csv",
    index_col="DATE",
    parse_dates=True,
)
energy.columns = ["Production"]
energy.index.freq = "MS"

print(f"Shape: {energy.shape}")
print(f"Date range: {energy.index[0].date()} to {energy.index[-1].date()}")

fig, ax = plt.subplots()
ax.plot(energy["Production"], linewidth=0.7)
ax.set_title("Energy Production Index — Monthly")
ax.set_ylabel("Index")
ax.set_xlabel("Date")
plt.tight_layout()
plt.show()

print("Clear upward trend and strong annual seasonality.")

In [ ]:
# Train/test split — hold out last 24 months
HORIZON = 24
train = energy.iloc[:-HORIZON]
test = energy.iloc[-HORIZON:]

print(f"Train: {len(train)} months ({train.index[0].date()} to {train.index[-1].date()})")
print(f"Test:  {len(test)} months ({test.index[0].date()} to {test.index[-1].date()})")

---
## 5. Baseline: Seasonal ARIMA

First, we fit a standard SARIMA with $m=12$ as a baseline.

In [ ]:
# Baseline SARIMA
sarima_model = SARIMAX(
    train["Production"],
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False,
)
sarima_result = sarima_model.fit(disp=False)

sarima_fc = sarima_result.get_forecast(steps=HORIZON)
sarima_pred = sarima_fc.predicted_mean

sarima_mae = mean_absolute_error(test["Production"], sarima_pred)
sarima_rmse = np.sqrt(mean_squared_error(test["Production"], sarima_pred))

print(f"Baseline SARIMA(1,1,1)(1,1,1)[12]:")
print(f"  AIC:  {sarima_result.aic:.2f}")
print(f"  MAE:  {sarima_mae:.2f}")
print(f"  RMSE: {sarima_rmse:.2f}")

---
## 6. Dynamic Harmonic Regression: ARIMA + Fourier

Now we replace the seasonal ARIMA component with Fourier terms as exogenous
variables.  The ARIMA part handles trend and short-range autocorrelation,
while the Fourier terms handle the seasonal pattern.

**Key point:** since the Fourier terms handle seasonality, we use a
**non-seasonal** ARIMA for the errors — no seasonal order needed.

In [ ]:
# Create Fourier features for the entire series, then split
fourier_all = create_fourier_features(energy.index, period=12, K=4)

fourier_train = fourier_all.iloc[:-HORIZON]
fourier_test = fourier_all.iloc[-HORIZON:]

print(f"Fourier features (K=4): {list(fourier_all.columns)}")
print(f"Train shape: {fourier_train.shape}")
print(f"Test shape:  {fourier_test.shape}")

In [ ]:
# Fit SARIMAX with Fourier features (non-seasonal ARIMA errors)
fourier_model = SARIMAX(
    endog=train["Production"],
    exog=fourier_train,
    order=(2, 1, 1),            # non-seasonal ARIMA for the errors
    # No seasonal_order — Fourier handles seasonality!
    enforce_stationarity=False,
    enforce_invertibility=False,
)
fourier_result = fourier_model.fit(disp=False)

print(f"Dynamic Harmonic Regression — ARIMA(2,1,1) + Fourier(K=4):")
print(f"AIC: {fourier_result.aic:.2f}")
print(fourier_result.summary().tables[1])

In [ ]:
# Forecast with Fourier features
fourier_fc = fourier_result.get_forecast(steps=HORIZON, exog=fourier_test)
fourier_pred = fourier_fc.predicted_mean
fourier_ci = fourier_fc.conf_int(alpha=0.05)

fourier_mae = mean_absolute_error(test["Production"], fourier_pred)
fourier_rmse = np.sqrt(mean_squared_error(test["Production"], fourier_pred))

print(f"Dynamic Harmonic Regression (K=4):")
print(f"  MAE:  {fourier_mae:.2f}")
print(f"  RMSE: {fourier_rmse:.2f}")

In [ ]:
# Visual comparison
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(test["Production"], label="Actual", color="tab:blue", linewidth=2)
ax.plot(sarima_pred, label="SARIMA(1,1,1)(1,1,1)[12]", color="tab:red", linestyle="--")
ax.plot(fourier_pred, label="ARIMA(2,1,1) + Fourier(K=4)", color="tab:green", linestyle="--")
ax.fill_between(
    fourier_ci.index, fourier_ci.iloc[:, 0], fourier_ci.iloc[:, 1],
    color="tab:green", alpha=0.1, label="95% CI (Fourier)",
)
ax.set_title("SARIMA vs Dynamic Harmonic Regression — Energy Production")
ax.set_ylabel("Production Index")
ax.legend()
plt.tight_layout()
plt.show()

---
## 7. Choosing K: Comparing Different Values

The choice of $K$ is a model selection problem.  We fit the dynamic harmonic
regression for $K = 1, 2, \ldots, 6$ and compare AIC and out-of-sample accuracy.

In [ ]:
# Compare K = 1 through 6
results = []
forecasts = {}

for K in range(1, 7):
    # Create Fourier features
    f_all = create_fourier_features(energy.index, period=12, K=K)
    f_train = f_all.iloc[:-HORIZON]
    f_test = f_all.iloc[-HORIZON:]

    # Fit model
    mod = SARIMAX(
        endog=train["Production"],
        exog=f_train,
        order=(2, 1, 1),
        enforce_stationarity=False,
        enforce_invertibility=False,
    ).fit(disp=False)

    # Forecast
    fc = mod.get_forecast(steps=HORIZON, exog=f_test)
    pred = fc.predicted_mean

    mae = mean_absolute_error(test["Production"], pred)
    rmse = np.sqrt(mean_squared_error(test["Production"], pred))

    results.append({
        "K": K,
        "Fourier_terms": 2 * K,
        "AIC": round(mod.aic, 2),
        "MAE": round(mae, 2),
        "RMSE": round(rmse, 2),
    })
    forecasts[K] = pred

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))
print(f"\nBest K by AIC:  {results_df.loc[results_df['AIC'].idxmin(), 'K']}")
print(f"Best K by MAE:  {results_df.loc[results_df['MAE'].idxmin(), 'K']}")
print(f"Best K by RMSE: {results_df.loc[results_df['RMSE'].idxmin(), 'K']}")

In [ ]:
# Plot AIC and RMSE vs K
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(results_df["K"], results_df["AIC"], marker="o", color="tab:blue")
axes[0].set_xlabel("K (number of Fourier pairs)")
axes[0].set_ylabel("AIC")
axes[0].set_title("AIC vs K")
axes[0].set_xticks(range(1, 7))

axes[1].plot(results_df["K"], results_df["RMSE"], marker="o", color="tab:red")
axes[1].set_xlabel("K (number of Fourier pairs)")
axes[1].set_ylabel("RMSE")
axes[1].set_title("Test RMSE vs K")
axes[1].set_xticks(range(1, 7))

plt.tight_layout()
plt.show()

print("Both AIC and RMSE typically improve with K up to a point,")
print("then plateau or worsen — find the elbow.")

In [ ]:
# Plot forecasts for different K values
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(test["Production"], label="Actual", color="black", linewidth=2)

colors = plt.cm.viridis(np.linspace(0.2, 0.9, 6))
for K, color in zip(range(1, 7), colors):
    ax.plot(forecasts[K], label=f"K={K}", color=color, linestyle="--", alpha=0.8)

ax.set_title("Forecasts with Different K Values")
ax.set_ylabel("Production Index")
ax.legend(ncol=2)
plt.tight_layout()
plt.show()

---
## 8. Why Fourier Terms? The Long-Period Advantage

The real power of dynamic harmonic regression shows up when the seasonal
period is **too long** for seasonal ARIMA.

| Seasonal period | Example | SARIMA parameters | Fourier (K=4) |
|-----------------|---------|-------------------|---------------|
| $m = 7$ (weekly) | Daily data, weekly cycle | Manageable | Also works |
| $m = 12$ (annual) | Monthly data | Standard | Also works |
| $m = 52$ (annual) | Weekly data | Difficult | Much better |
| $m = 365$ (annual) | Daily data | **Impractical** | **Essential** |

For $m = 365$, a seasonal ARIMA would need to estimate seasonal AR/MA
parameters at lag 365, 730, etc. — far too many parameters.  With Fourier
terms, we just need $2K$ exogenous columns (e.g., 8 columns for $K = 4$).

---
## 9. Application: Alcohol Sales with Fourier Seasonality

Let us also apply dynamic harmonic regression to the Alcohol Sales data
and compare with a standard seasonal SARIMA.

In [ ]:
# Load Alcohol Sales
alcohol = pd.read_csv(
    DATA_DIR / "Alcohol_Sales.csv",
    index_col="DATE",
    parse_dates=True,
)
alcohol.columns = ["Sales"]
alcohol.index.freq = "MS"

ALC_HORIZON = 24
alc_train = alcohol.iloc[:-ALC_HORIZON]
alc_test = alcohol.iloc[-ALC_HORIZON:]

# Baseline SARIMA
alc_sarima = SARIMAX(
    alc_train["Sales"],
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False,
).fit(disp=False)

alc_sarima_pred = alc_sarima.get_forecast(steps=ALC_HORIZON).predicted_mean
alc_sarima_rmse = np.sqrt(mean_squared_error(alc_test["Sales"], alc_sarima_pred))

print(f"Alcohol Sales — SARIMA baseline RMSE: {alc_sarima_rmse:.2f}")

In [ ]:
# Dynamic harmonic regression on Alcohol Sales — try K=1..6
alc_results = []

for K in range(1, 7):
    f_all = create_fourier_features(alcohol.index, period=12, K=K)
    f_train = f_all.iloc[:-ALC_HORIZON]
    f_test = f_all.iloc[-ALC_HORIZON:]

    mod = SARIMAX(
        endog=alc_train["Sales"],
        exog=f_train,
        order=(2, 1, 1),
        enforce_stationarity=False,
        enforce_invertibility=False,
    ).fit(disp=False)

    pred = mod.get_forecast(steps=ALC_HORIZON, exog=f_test).predicted_mean
    rmse = np.sqrt(mean_squared_error(alc_test["Sales"], pred))

    alc_results.append({"K": K, "AIC": round(mod.aic, 2), "RMSE": round(rmse, 2)})

alc_results_df = pd.DataFrame(alc_results)
print("Alcohol Sales — Dynamic Harmonic Regression:")
print(alc_results_df.to_string(index=False))
print(f"\nSARIMA baseline RMSE: {alc_sarima_rmse:.2f}")

---
## 10. Residual Check for Best Fourier Model

In [ ]:
# Re-fit the best K model and check residuals
best_K = int(results_df.loc[results_df["AIC"].idxmin(), "K"])
print(f"Best K by AIC: {best_K}")

best_fourier_all = create_fourier_features(energy.index, period=12, K=best_K)
best_fourier_train = best_fourier_all.iloc[:-HORIZON]

best_model = SARIMAX(
    endog=train["Production"],
    exog=best_fourier_train,
    order=(2, 1, 1),
    enforce_stationarity=False,
    enforce_invertibility=False,
).fit(disp=False)

fig = best_model.plot_diagnostics(figsize=(14, 10))
plt.suptitle(f"Residual Diagnostics — Fourier K={best_K}", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
lb = acorr_ljungbox(best_model.resid.dropna(), lags=[12, 24], return_df=True)
print(f"Ljung-Box test (K={best_K}):")
print(lb)

---
## Summary

- **Dynamic harmonic regression** uses Fourier sine/cosine pairs as
  exogenous variables in a SARIMAX model to capture seasonal patterns.
- Fourier terms are computed as $\sin(2\pi k t / m)$ and $\cos(2\pi k t / m)$
  for $k = 1, 2, \ldots, K$.
- **$K$ controls smoothness:** $K = 1$ gives a simple sine wave,
  $K = m/2$ can reproduce any seasonal shape exactly.
- **Advantage over seasonal ARIMA:** Fourier terms scale to any seasonal
  period — even $m = 365$ — with just $2K$ parameters.
- The ARIMA part of the model handles short-range autocorrelation and trend;
  **no seasonal order is needed** since Fourier handles seasonality.
- **Choose $K$** by minimising AIC or cross-validated out-of-sample error.
- Fourier features are **deterministic** — they can be computed for any
  future date, making them ideal exogenous variables for forecasting.
- This approach is especially valuable for **daily data with annual
  seasonality**, where $m = 365$ makes seasonal ARIMA impractical.

In [ ]:
print("This concludes Chapter 09: Dynamic Regression (SARIMAX).")
print()
print("Key takeaways:")
print("  1. Regression + ARIMA errors = dynamic regression (the 'X' in SARIMAX)")
print("  2. Exogenous variables must be known for the forecast horizon")
print("  3. Holidays and calendar effects are ideal exogenous variables")
print("  4. Fourier terms handle complex/long seasonality as exogenous regressors")
print("  5. Always compare against a pure SARIMA baseline")